# VisionGuard — Knowledge Distillation (FIXED)
## EfficientNet-B4 Teacher  →  MobileNetV3-Small Student

> **Patch notes (this version):** the original run produced `nan` loss from epoch 1, and the student's BatchNorm running stats got permanently corrupted by epoch 3, collapsing val QWK to ~0.16. Fixes applied: (1) loss math forced to fp32 with clamping, instead of computing inside fp16 `autocast`; (2) `BatchNorm1d` → `LayerNorm` in the student head (no persistent running stats to corrupt); (3) gradient clipping (`max_norm=5.0`); (4) non-finite batches are skipped instead of applied; (5) an early-stop guard if training still goes non-finite; (6) Phase-1 LR lowered 2e-4 → 1e-4. Changed cells: Student model, KDLoss, train/eval functions, training loop.

**Goal:** Compress the best trained model (best_model.pth, QWK 0.8431) into a
lightweight MobileNetV3-Small student via Knowledge Distillation, then export
the student to ONNX for subsequent TFLite conversion.

**Why KD instead of direct compression:**
The B4 teacher has 19M parameters and produces a file ~75 MB — too large and
slow for smooth inference on a phone. MobileNetV3-Small has ~2.5M parameters
(~7× smaller), with sub-1-second CPU inference on mid-range Android hardware.
KD trains the student to mimic the teacher's *soft outputs* (continuous
predicted scores), which carry richer ordinal information than one-hot labels.
Expected QWK after distillation: 0.82–0.84 (a small drop is acceptable since
deployment speed and file size are the binding constraints).

**Inputs required before running:**
1. APTOS 2019 competition dataset attached
   (`aptos2019-blindness-detection`)
2. best_model.pth uploaded as a Kaggle Dataset named `visionguard-model`
   and attached to this notebook.

**Notebook structure:**
- Cell 1  — Install and imports
- Cell 2  — Configuration
- Cell 3  — APTOS data loading
- Cell 4  — Preprocessing (Green channel + CLAHE, matching the teacher's training)
- Cell 5  — Dataset and DataLoaders
- Cell 6  — Teacher model: load and freeze
- Cell 7  — Student model: MobileNetV3-Small
- Cell 8  — KD loss function
- Cell 9  — Training loop
- Cell 10 — Run KD training
- Cell 11 — Training curves
- Cell 12 — Evaluation (student QWK vs teacher QWK)
- Cell 13 — ONNX export
- Cell 14 — Verify ONNX output matches PyTorch output
- Cell 15 — Summary

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 1 — Install & Imports             ║
# ╚══════════════════════════════════════════╝

# timm: EfficientNet-B4 teacher + MobileNetV3-Small student
# albumentations: augmentation pipeline
# onnx: ONNX export for TFLite conversion downstream
# onnxruntime: lightweight ONNX inference, used in Cell 14 to
#              verify the exported model without a full TF install

import subprocess, sys

def pip_install(pkg):
    """Install a package quietly; raise on failure."""
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'ERROR installing {pkg}:')
        print(result.stderr)
        raise RuntimeError(f'pip install {pkg} failed')

for pkg in ['timm', 'albumentations', 'onnx', 'onnxruntime']:
    pip_install(pkg)
    print(f'{pkg} ✅')

# ── Standard library ──────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight

# ── PyTorch ───────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# ── timm ──────────────────────────────────
import timm

# ── Albumentations ────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── ONNX ──────────────────────────────────
import onnx
import onnxruntime as ort

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')
print(f'timm   : {timm.__version__}')
print(f'onnx   : {onnx.__version__}')
print(f'ort    : {ort.__version__}')

In [ ]:
import os
print(os.listdir('/kaggle/input/notebooks/s567890/diabeticr-3'))

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 2 — Configuration                 ║
# ╚══════════════════════════════════════════╝

# ── Paths ─────────────────────────────────
# Teacher weights — uploaded as a Kaggle Dataset named 'visionguard-model'
# and saved with torch.save({'model_state': model.state_dict(), ...})
TEACHER_PATH = '/kaggle/input/notebooks/s567890/diabeticr-3/best_model.pth'

# APTOS dataset — same as the teacher's training data
APTOS_CSV      = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG_DIR  = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images/'

# Output paths
STUDENT_PTH    = '/kaggle/working/student_mobilenetv3.pth'
STUDENT_ONNX   = '/kaggle/working/student_mobilenetv3.onnx'

# ── Image ─────────────────────────────────
IMG_SIZE       = 384    # must match teacher's training resolution

# ── KD Hyperparameters ────────────────────
# Temperature (T): softens the teacher's output distribution before the
# student learns from it. T=4 is a standard starting value for regression KD —
# it makes the soft target smoother so the student learns relative grade
# relationships rather than just the hard predicted grade.
# Alpha: weight given to the distillation loss vs. the hard-label loss.
# 0.7 means 70% of learning comes from the teacher (soft), 30% from true labels.
TEMPERATURE    = 4.0
ALPHA          = 0.7    # distillation weight (1 - ALPHA = hard-label weight)

# ── Training ──────────────────────────────
# More epochs than the teacher used: the student is smaller and learns more
# slowly, especially in the first frozen phase.
KD_EPOCHS      = 35
BATCH_SIZE     = 32     # larger than teacher training (student is lighter)
LR             = 2e-4   # slightly higher than teacher; student has fewer params
WEIGHT_DECAY   = 1e-5
MIN_LR         = 1e-6
FOLD           = 0      # single fold for deployment focus
NUM_FOLDS      = 5

# ── Architecture names ────────────────────
TEACHER_NAME   = 'efficientnet_b4'
STUDENT_NAME   = 'mobilenetv3_small_100'  # timm model string

# ── Labels ────────────────────────────────
NUM_CLASSES    = 5
CLASS_NAMES    = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# Manually tuned class weights from the teacher's Grad-CAM failure analysis.
# These are kept fixed here (unlike the EyePACS experiments) because the
# student is being trained on the same APTOS distribution the teacher used.
CLASS_WEIGHTS_FIXED = [0.5, 2.0, 0.8, 5.0, 4.5]

print('Config loaded ✅')
print(f'Teacher: {TEACHER_PATH}')
print(f'Student: {STUDENT_NAME}')
print(f'KD epochs: {KD_EPOCHS} | LR: {LR} | T: {TEMPERATURE} | α: {ALPHA}')
print(f'Class weights (fixed, APTOS-tuned): {CLASS_WEIGHTS_FIXED}')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 3 — Load APTOS Data               ║
# ╚══════════════════════════════════════════╝

df = pd.read_csv(APTOS_CSV)
print(f'Total images: {len(df)}')

# Stratified fold — same setup the teacher used so the student sees
# the same training/validation split. Fold 0 is used throughout.
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df, df['diagnosis'])):
    if fold_idx == FOLD:
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df   = df.iloc[val_idx].reset_index(drop=True)
        break

print(f'\nFold {FOLD} split:')
print(f'  Train: {len(train_df)} images')
print(f'  Val  : {len(val_df)} images')

print('\nClass distribution (train):')
for g, c in train_df['diagnosis'].value_counts().sort_index().items():
    bar = '█' * (c // 40)
    pct = c / len(train_df) * 100
    print(f'  Grade {g} ({CLASS_NAMES[g]:>16s}): {c:4d} ({pct:4.1f}%)  {bar}')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 4 — Preprocessing Functions       ║
# ╚══════════════════════════════════════════╝
# Must exactly match the teacher's training preprocessing.
# Any difference here means the student learns from images that look
# different from what the teacher was trained on — the soft targets
# would no longer be meaningful for those inputs.

def preprocess_image(img_path):
    """
    Green channel + CLAHE pipeline used for teacher training.
    1. Read image
    2. Extract green channel (highest signal-to-noise for DR lesions)
    3. Apply CLAHE (local contrast normalisation, camera-agnostic)
    4. Stack 3x to match EfficientNet's expected 3-channel input
    """
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {img_path}')

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Green channel: highest SNR for microaneurysms vs red (oversat) / blue (noisy)
    green = img[:, :, 1]

    # CLAHE: local contrast enhancement, tile-wise so bright regions
    # are not overexposed while correcting dark ones.
    # clipLimit=2.0 is the validated default for retinal imaging.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Stack 3x: EfficientNet expects (H, W, 3); stacking preserves
    # pretrained-weight compatibility (same values in all 3 channels).
    stacked = np.stack([enhanced, enhanced, enhanced], axis=-1)
    return stacked   # (H, W, 3) uint8

# ── Quick smoke test ──────────────────────
sample_path = APTOS_IMG_DIR + df['id_code'].iloc[0] + '.png'
test_img    = preprocess_image(sample_path)
print(f'Preprocessing ✅')
print(f'  Sample input shape : {cv2.imread(sample_path).shape}')
print(f'  Preprocessed shape : {test_img.shape}  (dtype={test_img.dtype})')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 5 — Dataset & DataLoaders         ║
# ╚══════════════════════════════════════════╝

# ── Augmentations ─────────────────────────
# Keep augmentations consistent with teacher training.
# The student inherits the teacher's knowledge, not the teacher's data
# augmentation strategy, but consistent augmentation means both models
# see similarly distributed inputs during the distillation run.
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=20, p=0.4),
    A.OneOf([
        A.GaussNoise(p=1),
        A.GaussianBlur(blur_limit=3, p=1),
    ], p=0.2),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ── Dataset class ─────────────────────────
class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        image    = preprocess_image(img_path)   # (H, W, 3) uint8

        if self.transform:
            image = self.transform(image=image)['image']   # (3, H, W) float tensor

        # Long label for QWK evaluation; KD loss converts to float internally
        label = torch.tensor(row['diagnosis'], dtype=torch.long)
        return image, label

train_ds = APTOSDataset(train_df, APTOS_IMG_DIR, transform=train_transform)
val_ds   = APTOSDataset(val_df,   APTOS_IMG_DIR, transform=val_transform)

# drop_last=True on train loader: prevents a batch of size 1 reaching
# BatchNorm1d, which cannot compute statistics from a single sample.
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2,
    pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2,
    pin_memory=True
)

print(f'Train: {len(train_ds)} images, {len(train_loader)} batches')
print(f'Val  : {len(val_ds)} images, {len(val_loader)} batches')
print(f'Batch size: {BATCH_SIZE} | Image size: {IMG_SIZE}x{IMG_SIZE}')
print('DataLoaders ready ✅')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 6 — Teacher Model                 ║
# ╚══════════════════════════════════════════╝

# Exact same DRModel definition used during teacher training.
# Must not change anything here — if the architecture differs from what
# was used to produce best_model.pth, the state_dict will fail to load.
class DRModel(nn.Module):
    def __init__(self, model_name='efficientnet_b4', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained  = pretrained,
            num_classes = 0,
            global_pool = 'avg'   # timm pools internally -> (B, 1792)
        )
        in_features = self.backbone.num_features  # 1792 for B4

        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.4),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 1)     # single output -- ordinal regression
        )

    def forward(self, x):
        return self.head(self.backbone(x))   # (B, 1)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_last_blocks(self, n=2):
        for block in list(self.backbone.blocks)[-n:]:
            for p in block.parameters(): p.requires_grad = True
    def unfreeze_all(self):
        for p in self.backbone.parameters(): p.requires_grad = True


# ── Load checkpoint ───────────────────────
if not os.path.exists(TEACHER_PATH):
    raise FileNotFoundError(
        f'Teacher weights not found at {TEACHER_PATH}\n'
        f'Upload best_model.pth as a Kaggle Dataset named '
        f'"visionguard-model" and add it as an input to this notebook.'
    )

# weights_only=False: the checkpoint also stores epoch/kappa alongside
# the state_dict; PyTorch 2.6+ requires this flag to be explicit.
ckpt = torch.load(TEACHER_PATH, map_location=DEVICE, weights_only=False)

# The checkpoint was saved as {'model_state': ..., 'epoch': ..., 'kappa': ...}
# Fall back to loading the dict directly in case it was saved bare.
if isinstance(ckpt, dict) and 'model_state' in ckpt:
    state_dict = ckpt['model_state']
    saved_epoch = ckpt.get('epoch', 'unknown')
    saved_kappa = ckpt.get('kappa', 'unknown')
else:
    state_dict  = ckpt
    saved_epoch = 'unknown'
    saved_kappa = 'unknown'

teacher = DRModel(model_name=TEACHER_NAME, pretrained=False).to(DEVICE)
teacher.load_state_dict(state_dict)

# Freeze all teacher parameters permanently — it never updates during KD.
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()

n_teacher = sum(p.numel() for p in teacher.parameters())
print(f'Teacher loaded ✅')
print(f'  Saved at epoch : {saved_epoch}')
print(f'  Saved QWK      : {saved_kappa}')
print(f'  Parameters     : {n_teacher:,}')

# ── Sanity check ──────────────────────────
with torch.no_grad():
    dummy  = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    t_out  = teacher(dummy)
print(f'  Output shape   : {t_out.shape}  <- should be (2, 1)')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  DIAGNOSTIC — Check teacher checkpoint   ║
# ╚══════════════════════════════════════════╝
# Run this once, right after the teacher is loaded, BEFORE starting KD
# training. It checks whether best_model.pth itself contains any nan/inf
# weights -- if it does, every single forward pass through the teacher
# will be nan/inf, which exactly matches "distil_loss=nan on every batch".

bad_params = []
for name, p in teacher.named_parameters():
    if not torch.isfinite(p).all():
        n_bad = (~torch.isfinite(p)).sum().item()
        bad_params.append((name, n_bad, p.numel()))

if bad_params:
    print(f'⛔ FOUND non-finite weights in {len(bad_params)} parameter tensor(s):')
    for name, n_bad, total in bad_params:
        print(f'   {name}: {n_bad}/{total} non-finite values')
    print()
    print('This confirms best_model.pth is corrupted. The checkpoint needs')
    print('to be re-saved from a healthy epoch of the ORIGINAL teacher training')
    print('run (not this KD notebook) -- check if you have an earlier-epoch')
    print('checkpoint, or re-run teacher training with the same fp32-loss /')
    print('grad-clipping fixes used in the student fixes.')
else:
    print('✅ All teacher weights are finite. Checkpoint itself is clean.')
    print('   Running one more check: forward pass on real validation data...')

    teacher.eval()
    with torch.no_grad():
        sample_imgs, _ = next(iter(val_loader))
        sample_imgs = sample_imgs.to(DEVICE)

        # fp32 forward
        out_fp32 = teacher(sample_imgs)
        print(f'   fp32 teacher output on real batch: finite={torch.isfinite(out_fp32).all().item()}, '
              f'range=[{out_fp32.min().item():.3f}, {out_fp32.max().item():.3f}]')

        # fp16/autocast forward (this is what train_epoch_kd actually uses)
        with autocast():
            out_amp = teacher(sample_imgs)
        print(f'   autocast teacher output on real batch: finite={torch.isfinite(out_amp).all().item()}, '
              f'range=[{out_amp.float().min().item():.3f}, {out_amp.float().max().item():.3f}]')

        if torch.isfinite(out_fp32).all() and not torch.isfinite(out_amp).all():
            print()
            print('⚠️  The checkpoint is clean in fp32 but produces nan/inf under autocast.')
            print('   Fix: run the TEACHER forward pass in fp32 (outside autocast),')
            print('   only keep the STUDENT forward pass under autocast. See patch below.')


In [ ]:
class StudentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            STUDENT_NAME,
            pretrained  = True,
            num_classes = 0,
            global_pool = 'avg'
        )

        # Get the real feature size by running a dummy input through
        # the backbone — timm's num_features property is unreliable for
        # some MobileNetV3 variants before a forward pass.
        with torch.no_grad():
            dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
            n = self.backbone(dummy).shape[1]
        print(f'  MobileNetV3-Small actual feature size: {n}')

        # FIX: LayerNorm instead of BatchNorm1d.
        # BatchNorm1d keeps running_mean/running_var that get UPDATED even
        # during a bad batch. If one fp16 batch produces inf/nan activations,
        # those NaNs get folded permanently into running stats, and every
        # batch afterward is corrupted -- this is what caused QWK to collapse
        # to 0 from epoch 3 onward in the original run. LayerNorm normalizes
        # per-sample with no persistent state, so a single bad batch cannot
        # poison future batches.
        self.head = nn.Sequential(
            nn.LayerNorm(n),
            nn.Dropout(0.2),
            nn.Linear(n, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


student = StudentModel().to(DEVICE)

n_student = sum(p.numel() for p in student.parameters())
n_teacher = sum(p.numel() for p in teacher.parameters())
print(f'Student created ✅  ({STUDENT_NAME})')
print(f'  Teacher params : {n_teacher:,}')
print(f'  Student params : {n_student:,}')
print(f'  Compression    : {n_teacher/n_student:.1f}x smaller')

student.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    s_out = student(dummy)
print(f'  Output shape   : {s_out.shape}  <- should be (2, 1)')
student.train()


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 8 — Knowledge Distillation Loss   ║
# ╚══════════════════════════════════════╝

class KDLoss(nn.Module):
    """
    Combined Knowledge Distillation + Hard-label loss for ordinal regression.

    Components:
    1. Distillation loss (weight=alpha):
       MSE between student output and teacher's SOFT output.
       The teacher's output is divided by Temperature T before comparison —
       a higher T spreads the prediction distribution so the student learns
       'how wrong' a miss is, not just the hard predicted grade.
       Temperature is applied to both teacher and student consistently.

    2. Hard-label loss (weight=1-alpha):
       Weighted MSE between student output and the true integer grade.
       Uses CLASS_WEIGHTS_FIXED (same as teacher training) to ensure the
       student also inherits the penalty structure on Grades 3 and 4.

    Combined loss = alpha * distil_loss + (1 - alpha) * hard_loss

    FIX (numerical stability):
    All loss math below now runs strictly in fp32. The original version
    was called from inside an autocast() block, so the MSE on /T-scaled
    fp16 tensors could overflow to inf -> nan, and that nan got folded
    into BatchNorm running stats permanently. Forcing fp32 here removes
    that overflow path. (We also clamp s/t to a sane range as a second
    line of defense in case the backbone itself ever produces extreme
    values early in training.)
    """
    def __init__(self, class_weights, device,
                 temperature=TEMPERATURE, alpha=ALPHA):
        super().__init__()
        self.T       = temperature
        self.alpha   = alpha
        self.weights = torch.tensor(
            class_weights, dtype=torch.float32, device=device
        )

    def forward(self, student_out, teacher_out, true_labels):
        # student_out  : (B, 1) raw student scores
        # teacher_out  : (B, 1) raw teacher scores (detached, no grad)
        # true_labels  : (B,)   integer true grades (long)

        # FIX: force fp32 regardless of what dtype these arrive in
        # (e.g. fp16 from an autocast block upstream).
        s = student_out.squeeze(1).float()   # (B,)
        t = teacher_out.squeeze(1).float()   # (B,)

        # FIX: clamp to a generous but finite range. Early in training,
        # an untrained head can emit large outliers; clamping prevents
        # those from squaring into inf under any precision.
        s = torch.clamp(s, -50.0, 50.0)
        t = torch.clamp(t, -50.0, 50.0)

        # Distillation: compare temperature-scaled outputs
        # Both divided by T so the relative distances are preserved.
        distil_loss = F.mse_loss(s / self.T, t / self.T)

        # Hard-label: weighted MSE against true grades
        labels_f       = true_labels.float()
        sample_weights = self.weights[true_labels.clamp(0, 4)]
        hard_loss      = (sample_weights * (s - labels_f) ** 2).mean()

        loss = self.alpha * distil_loss + (1.0 - self.alpha) * hard_loss

        # FIX: surface a clear error if something is still producing
        # non-finite loss, instead of silently corrupting training.
        if not torch.isfinite(loss):
            print(f'  ⚠️ non-finite loss component | distil={distil_loss.item():.4f} '
                  f'hard={hard_loss.item():.4f}')

        return loss


kd_criterion = KDLoss(CLASS_WEIGHTS_FIXED, DEVICE)
print(f'KD Loss defined ✅ (fp32-safe)')
print(f'  Temperature (T) : {TEMPERATURE}')
print(f'  Alpha           : {ALPHA} (distillation) + {1-ALPHA} (hard label)')
print(f'  Class weights   : {CLASS_WEIGHTS_FIXED}')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 9 — Training & Eval Functions     ║
# ╚══════════════════════════════════════╝

def train_epoch_kd(student, teacher, loader, criterion,
                   optimizer, scaler, device, max_grad_norm=5.0):
    student.train()
    teacher.eval()   # teacher stays in eval mode throughout
    total_loss   = 0.0
    n_batches_ok = 0
    n_skipped    = 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)   # long for class-weight indexing

        optimizer.zero_grad()

        # FIX: teacher forward runs in fp32, fully outside autocast.
        # The teacher is frozen and only ever does inference here -- there is
        # no training-speed reason to run it in fp16. EfficientNet-B4's
        # SE-blocks (sigmoid-gated attention) are a known overflow risk under
        # fp16 autocast independent of the loss math; running the teacher in
        # fp32 removes that risk entirely regardless of what the diagnostic
        # cell above finds.
        with torch.no_grad():
            t_out = teacher(imgs.float())   # (B, 1), fp32

        with autocast():
            # Student forward (this one benefits from fp16 speed since it trains)
            s_out = student(imgs)       # (B, 1)

        # FIX: loss computed OUTSIDE autocast, forced to fp32 inside KDLoss.
        # This is the main fix for the nan-from-epoch-1 bug: the original
        # code computed (s/T)^2-style MSE terms INSIDE the fp16 autocast
        # region, which could overflow to inf -> nan. KDLoss.forward()
        # casts to float32 and clamps before squaring, so this is safe now.
        loss = criterion(s_out, t_out, labels)

        # FIX: if a batch still produces a non-finite loss (shouldn't happen
        # now, but defend anyway), skip the optimizer step entirely so we
        # never let nan/inf gradients update BatchNorm/LayerNorm parameters
        # or corrupt running statistics.
        if not torch.isfinite(loss):
            n_skipped += 1
            continue

        scaler.scale(loss).backward()

        # FIX: gradient clipping. Unscale first (required when using
        # GradScaler) so clip_grad_norm_ sees true gradient magnitudes.
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()

        total_loss   += loss.item() * imgs.size(0)
        n_batches_ok += 1

        preds = torch.clamp(
            torch.round(s_out.detach().cpu().float().squeeze(1)), 0, 4
        ).long().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    if n_skipped > 0:
        print(f'    ⚠️  skipped {n_skipped} non-finite batch(es) this epoch')

    denom = max(n_batches_ok, 1) * loader.batch_size
    qwk   = cohen_kappa_score(all_labels, all_preds, weights='quadratic') \
            if len(all_preds) > 0 else 0.0
    return total_loss / max(len(all_preds), 1), qwk


def val_epoch_student(student, loader, class_weights, device):
    """
    Validation uses only the hard-label weighted MSE (no teacher needed) —
    we want to know how well the student predicts true grades, not how
    closely it mimics the teacher on the validation set.
    """
    student.eval()
    weights    = torch.tensor(class_weights, dtype=torch.float32, device=device)
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)

            with autocast():
                s_out = student(imgs).squeeze(1)   # (B,)

            # FIX: force fp32 for the loss math here too, same reasoning
            # as the training loss above.
            s_out_f32      = s_out.float()
            labels_f       = labels.float()
            sample_weights = weights[labels.clamp(0, 4)]
            loss           = (sample_weights * (s_out_f32 - labels_f) ** 2).mean()

            total_loss += loss.item() * imgs.size(0)

            preds = torch.clamp(
                torch.round(s_out_f32.cpu()), 0, 4
            ).long().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return total_loss / len(loader.dataset), qwk


print('Training functions defined ✅ (fp32 loss, grad clipping, NaN-skip)')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 10 — Run KD Training              ║
# ╚══════════════════════════════════════╝

# Gradual unfreezing for the student backbone — same three-phase logic that
# worked for the teacher, now applied to the student's lighter backbone.
# Phase 1 (epochs 1-5)  : student head only; backbone frozen
# Phase 2 (epochs 6-15) : unfreeze last 2 blocks of student backbone
# Phase 3 (epoch 16+)   : full student backbone unfrozen

# FIX: lower phase-1 LR slightly (2e-4 -> 1e-4). A freshly initialized head
# sitting on top of a frozen pretrained backbone is the most fragile point
# in training; combined with the fp32-loss + grad-clip fixes above this
# gives the head a gentler start. The cosine schedule still anneals it down
# from here over KD_EPOCHS, so this does not change the overall trajectory
# much, just the size of the first few steps.
PHASE1_LR = 1e-4

optimizer = optim.Adam(student.parameters(),
                       lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=KD_EPOCHS, eta_min=MIN_LR
)
scaler    = GradScaler()

history  = {k: [] for k in
             ['train_loss', 'val_loss', 'train_qwk', 'val_qwk']}
best_qwk = -1.0
best_ep  = 0

# Freeze the student backbone initially (train head only in Phase 1)
for p in student.backbone.parameters():
    p.requires_grad = False

print(f'Starting KD training — {KD_EPOCHS} epochs')
print(f'Phase 1: head only (backbone frozen), LR={PHASE1_LR:.1e}')
print('=' * 70)

for epoch in range(1, KD_EPOCHS + 1):

    # ── Unfreezing schedule ─────────────────────────────────────────────
    if epoch == 6:
        # Unfreeze last 2 blocks of MobileNetV3-Small backbone
        blocks = list(student.backbone.blocks)
        for block in blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True
        print('\nPhase 2: Unfroze last 2 blocks of student backbone')

    if epoch == 16:
        for p in student.backbone.parameters():
            p.requires_grad = True
        print('\nPhase 3: Full student backbone unfrozen')

    # ── Train ──────────────────────────────────────────────────
    t_loss, t_qwk = train_epoch_kd(
        student, teacher, train_loader, kd_criterion,
        optimizer, scaler, DEVICE
    )

    # ── Validate ───────────────────────────────────────────────
    v_loss, v_qwk = val_epoch_student(
        student, val_loader, CLASS_WEIGHTS_FIXED, DEVICE
    )

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_qwk'].append(t_qwk)
    history['val_qwk'].append(v_qwk)

    # FIX: extra safety net — if something still goes non-finite mid-run,
    # stop immediately with a clear message instead of burning through
    # the remaining epochs on a corrupted model (this is what silently
    # happened for 32 wasted epochs in the original run).
    if not (np.isfinite(t_loss) and np.isfinite(v_loss)):
        print(f'\n⛔ Stopping early at epoch {epoch}: non-finite loss detected.')
        print(f'   train_loss={t_loss}, val_loss={v_loss}')
        print(f'   This means the fixes above were not enough — try lowering '
              f'PHASE1_LR further (e.g. 5e-5) or LR overall, or reduce '
              f'max_grad_norm to 1.0.')
        break

    saved_str = ''
    if v_qwk > best_qwk:
        best_qwk = v_qwk
        best_ep  = epoch
        torch.save({
            'model_state': student.state_dict(),
            'epoch'      : epoch,
            'qwk'        : best_qwk,
            'model_name' : STUDENT_NAME,
        }, STUDENT_PTH)
        saved_str = '  ✅ BEST'

    print(
        f'Epoch {epoch:02d}/{KD_EPOCHS} | LR: {current_lr:.1e} | '
        f'Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | '
        f'Train QWK: {t_qwk:.4f} | Val QWK: {v_qwk:.4f}'
        + saved_str
    )

print(f'\nKD Training complete!')
print(f'Best Student Val QWK : {best_qwk:.4f}  (epoch {best_ep})')
print(f'Teacher QWK (baseline): 0.8431')
print(f'Acceptable drop       : ~0.01-0.02 for a {n_teacher/n_student:.1f}x compression')


In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 11 — Training Curves              ║
# ╚══════════════════════════════════════════╝

epochs_x = list(range(1, len(history['train_loss']) + 1))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_x, history['train_loss'], 'b-o', markersize=3, label='Train')
axes[0].plot(epochs_x, history['val_loss'],   'r-o', markersize=3, label='Val')
axes[0].set_title('KD Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# QWK
axes[1].plot(epochs_x, history['train_qwk'], 'b-o', markersize=3, label='Train QWK')
axes[1].plot(epochs_x, history['val_qwk'],   'r-o', markersize=3, label='Val QWK')
axes[1].axhline(y=best_qwk,   color='orange', linestyle='--',
                label=f'Best student QWK={best_qwk:.4f}')
axes[1].axhline(y=0.8431, color='green', linestyle=':',
                label='Teacher QWK=0.8431')
axes[1].set_title('Quadratic Weighted Kappa', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('QWK')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('MobileNetV3-Small Student — KD Training Curves',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/kd_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 12 — Final Evaluation             ║
# ╚══════════════════════════════════════════╝

# Load best student checkpoint
ckpt = torch.load(STUDENT_PTH, map_location=DEVICE, weights_only=False)
student.load_state_dict(ckpt['model_state'])
print(f'Loaded best student from epoch {ckpt["epoch"]} (QWK={ckpt["qwk"]:.4f})')

student.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Evaluating student'):
        imgs = imgs.to(DEVICE)
        with autocast():
            s_out = student(imgs).squeeze(1)
        preds = torch.clamp(
            torch.round(s_out.cpu().float()), 0, 4
        ).long().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc   = (all_preds == all_labels).mean()
kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

print(f'\n[Student — MobileNetV3-Small]')
print(f'  Val Accuracy : {acc*100:.2f}%')
print(f'  Val QWK      : {kappa:.4f}')
print(f'\n[Teacher — EfficientNet-B4]')
print(f'  Val QWK      : 0.8431  (from best_model.pth)')
print(f'\nQWK drop after {n_teacher/n_student:.1f}x compression: '
      f'{0.8431 - kappa:.4f}')

print('\nPer-class accuracy:')
for cls in range(5):
    mask    = all_labels == cls
    cls_acc = (all_preds[mask] == cls).mean() if mask.sum() > 0 else 0.0
    bar     = '█' * int(cls_acc * 20)
    print(f'  Grade {cls} ({CLASS_NAMES[cls]:<16}): {cls_acc*100:5.1f}%  {bar}')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 13 — Export to ONNX               ║
# ╚══════════════════════════════════════════╝
# Converts the student PyTorch model to ONNX format.
# ONNX is the intermediate format for subsequent TFLite conversion
# (done separately in a Colab notebook using onnx2tf).

student.eval()

# Dummy input matching the model's expected input shape.
# Batch dim is set to 1 for mobile inference.
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

# Verify the model runs before attempting export
with torch.no_grad():
    test_out = student(dummy_input)
print(f'Pre-export sanity check: output = {test_out.item():.4f} ✅')

# Export
# opset_version=11: widely supported by onnx2tf and TFLite converters.
# dynamic_axes on the batch dimension allows the ONNX model to accept
# any batch size at inference time (1 for mobile, larger for testing).
torch.onnx.export(
    student,
    dummy_input,
    STUDENT_ONNX,
    opset_version  = 11,
    input_names    = ['input'],
    output_names   = ['output'],
    dynamic_axes   = {
        'input' : {0: 'batch_size'},
        'output': {0: 'batch_size'},
    },
    do_constant_folding = True,   # folds constant sub-expressions for speed
)
print(f'ONNX export complete ✅')

# ── Validate ONNX model ───────────────────
onnx_model = onnx.load(STUDENT_ONNX)
onnx.checker.check_model(onnx_model)
print(f'ONNX model validation passed ✅')

size_mb = os.path.getsize(STUDENT_ONNX) / (1024 * 1024)
print(f'ONNX file size: {size_mb:.1f} MB')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 14 — Verify ONNX vs PyTorch       ║
# ╚══════════════════════════════════════════╝
# Runs the same 5 validation images through both the PyTorch model and the
# exported ONNX model and compares outputs. If the max difference is below
# 1e-4, the export is numerically correct and ready for TFLite conversion.

student.eval()

session = ort.InferenceSession(STUDENT_ONNX)
input_name = session.get_inputs()[0].name

max_diff = 0.0
n_check  = min(5, len(val_ds))

for i in range(n_check):
    img_tensor, label = val_ds[i]
    img_batch = img_tensor.unsqueeze(0).to(DEVICE)   # (1, 3, H, W)

    # PyTorch output
    with torch.no_grad():
        pt_out = student(img_batch).item()

    # ONNX output
    onnx_in   = img_tensor.unsqueeze(0).numpy()       # (1, 3, H, W) float32
    onnx_out  = session.run(None, {input_name: onnx_in})[0]
    onnx_val  = float(onnx_out[0, 0])

    diff = abs(pt_out - onnx_val)
    max_diff = max(max_diff, diff)
    print(f'  Image {i+1}: PyTorch={pt_out:.5f}  ONNX={onnx_val:.5f}  '
          f'diff={diff:.2e}  true_grade={label.item()}')

print()
threshold = 1e-4
if max_diff < threshold:
    print(f'✅ Max difference {max_diff:.2e} < {threshold} — ONNX export is numerically correct')
else:
    print(f'⚠️  Max difference {max_diff:.2e} exceeds {threshold} — check export settings')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELL 15 — Summary                      ║
# ╚══════════════════════════════════════════╝

student_size = os.path.getsize(STUDENT_PTH) / (1024 * 1024)
onnx_size    = os.path.getsize(STUDENT_ONNX) / (1024 * 1024)

print('=' * 60)
print('KNOWLEDGE DISTILLATION — FINAL SUMMARY')
print('=' * 60)
print()
print(f'Teacher (EfficientNet-B4):')
print(f'  QWK        : 0.8431')
print(f'  Parameters : {n_teacher:,}  (~75 MB .pth)')
print()
print(f'Student (MobileNetV3-Small):')
print(f'  QWK        : {best_qwk:.4f}')
print(f'  Parameters : {n_student:,}')
print(f'  .pth size  : {student_size:.1f} MB')
print(f'  .onnx size : {onnx_size:.1f} MB')
print(f'  Compression: {n_teacher/n_student:.1f}x smaller than teacher')
print(f'  QWK drop   : {0.8431 - best_qwk:.4f}')
print()
print(f'Next step:')
print(f'  Download {STUDENT_ONNX}')
print(f'  Convert to TFLite INT8 using the Colab onnx2tf pipeline')
print(f'  Expected TFLite size: ~4-8 MB (after INT8 quantization)')
print('=' * 60)